# UCI Datasets Visualization

This notebook visualizes the distributions of the UCI datasets used in the LSS-boost experiments.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style and size
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

In [ ]:
# Add the parent directory to the path to import from utils
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))  

# Define the dataset loaders (copied from UCI_xglssboost_autoselect_reverse_HP_single_run.py)
dataset_name_to_loader = {
    "Boston Housing": lambda: pd.read_csv(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/housing/housing.data",
        header=None,
        delim_whitespace=True,
    ),
    "Concrete Compression Strength": lambda: pd.read_excel(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"
    ),
    "Energy Efficiency": lambda: pd.read_excel(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/00242/ENB2012_data.xlsx"
    ).iloc[:, :-1],
    "Kin8nm": lambda: pd.read_csv("../ngboost/data/uci/kin8nm.csv"),
    "Naval Propulsion": lambda: pd.read_csv(
        "../ngboost/data/uci/naval-propulsion.txt", delim_whitespace=True, header=None
    ).iloc[:, :-1],
    "Combined Cycle Power Plant": lambda: pd.read_excel("../ngboost/data/uci/power-plant.xlsx"),
    "Protein Structure": lambda: pd.read_csv("../ngboost/data/uci/protein.csv")[
        ["F1", "F2", "F3", "F4", "F5", "F6", "F7", "F8", "F9", "RMSD"]
    ],
    "Wine Quality Red": lambda: pd.read_csv(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv",
        delimiter=";",
    ),
    "Yacht Hydrodynamics": lambda: pd.read_csv(
        "http://archive.ics.uci.edu/ml/machine-learning-databases/00243/yacht_hydrodynamics.data",
        header=None,
        delim_whitespace=True,
    ),
    "Year Prediciton MSD": lambda: pd.read_csv("../ngboost/data/uci/YearPredictionMSD.txt").iloc[:, ::-1],
}

dataset_list = ["Boston Housing", "Concrete Compression Strength", "Energy Efficiency", "Kin8nm", 
                "Naval Propulsion", "Combined Cycle Power Plant", "Protein Structure", 
                "Wine Quality Red", "Yacht Hydrodynamics", "Year Prediciton MSD"]

## Basic Dataset Information

Let's collect some basic information about each dataset:

In [ ]:
def get_dataset_info(dataset_name):
    try:
        # Load dataset
        data = dataset_name_to_loader[dataset_name]()
        
        # Extract target variable (last column by default)
        X, y = data.iloc[:, :-1], data.iloc[:, -1]
        
        # Collect info
        info = {
            "Dataset Name": dataset_name,
            "Number of Samples": len(data),
            "Number of Features": X.shape[1],
            "Target Mean": y.mean(),
            "Target Std": y.std(),
            "Target Min": y.min(),
            "Target Max": y.max(),
            "Target Skewness": y.skew(),
            "Target Kurtosis": y.kurt()
        }
        return info, data
    except Exception as e:
        print(f"Error loading {dataset_name}: {str(e)}")
        return None, None

# Collect information for all datasets
dataset_info_list = []
datasets_data = {}

for dataset_name in dataset_list:
    print(f"Loading {dataset_name}...")
    info, data = get_dataset_info(dataset_name)
    if info:
        dataset_info_list.append(info)
        datasets_data[dataset_name] = data

# Convert to dataframe for nice display
dataset_info_df = pd.DataFrame(dataset_info_list)
dataset_info_df

## Visualizing Target Distributions

Let's visualize the distribution of the target variable for each dataset:

In [ ]:
# Create a grid of histograms for target distributions
n_datasets = len(datasets_data)
n_cols = 2
n_rows = (n_datasets + 1) // 2

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, (dataset_name, data) in enumerate(datasets_data.items()):
    y = data.iloc[:, -1]
    
    # Plot histogram with KDE
    sns.histplot(y, kde=True, ax=axes[i])
    
    # Add QQ plot as an insert to check normality
    from scipy import stats
    ax_inset = axes[i].inset_axes([0.6, 0.6, 0.35, 0.35])
    stats.probplot(y, plot=ax_inset)
    ax_inset.set_title('QQ Plot', fontsize=8)
    ax_inset.tick_params(axis='both', which='both', labelsize=6)
    
    # Set titles and labels
    axes[i].set_title(f"{dataset_name} - Target Distribution")
    axes[i].set_xlabel('Target Value')
    axes[i].set_ylabel('Frequency')
    
    # Add skewness and kurtosis as text
    skew = y.skew()
    kurt = y.kurtosis()
    stats_text = f"Skewness: {skew:.2f}\nKurtosis: {kurt:.2f}"
    axes[i].text(0.05, 0.95, stats_text, transform=axes[i].transAxes, 
                 verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## Feature Distributions for Each Dataset

Let's examine the distribution of features in each dataset:

In [ ]:
# Function to plot feature distributions for a dataset
def plot_feature_distributions(dataset_name, data):
    print(f"\n## {dataset_name} Feature Distributions")
    
    # Get feature names
    if all(isinstance(col, int) for col in data.columns):
        # If columns are integers (no header), rename them
        data.columns = [f'Feature_{i}' if i < len(data.columns)-1 else 'Target' for i in range(len(data.columns))]
    
    X = data.iloc[:, :-1]
    n_features = X.shape[1]
    
    # Limit to first 10 features if there are too many
    if n_features > 10:
        print(f"Dataset has {n_features} features. Showing only the first 10.")
        X = X.iloc[:, :10]
        n_features = 10
    
    # Create subplot grid
    n_cols = 3
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    # Plot each feature
    for i, col in enumerate(X.columns):
        sns.histplot(X[col], kde=True, ax=axes[i])
        axes[i].set_title(f"{col} Distribution")
    
    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # Correlation with target
    y = data.iloc[:, -1]
    correlations = X.corrwith(y).sort_values(ascending=False)
    
    plt.figure(figsize=(10, 6))
    correlations.plot(kind='bar')
    plt.title(f"{dataset_name}: Feature Correlations with Target")
    plt.xlabel('Feature')
    plt.ylabel('Correlation with Target')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Plot for each dataset (limited to first 3 to avoid notebook getting too large)
for i, (dataset_name, data) in enumerate(list(datasets_data.items())[:3]):
    plot_feature_distributions(dataset_name, data)

## Target Distribution Analysis with Potential Statistical Distributions

Let's fit several statistical distributions to the target variables to see which one fits best:

In [ ]:
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# List of distributions to try
distributions = [stats.norm, stats.t, stats.gamma, stats.lognorm, 
                 stats.weibull_min, stats.gumbel_r, stats.laplace]

# Function to fit distributions and calculate AIC
def fit_distributions(data):
    results = []
    
    for dist in distributions:
        try:
            # Fit distribution
            params = dist.fit(data)
            
            # Calculate log likelihood
            log_likelihood = np.sum(dist.logpdf(data, *params))
            
            # Calculate AIC: -2*log_likelihood + 2*k (k=number of parameters)
            k = len(params)
            aic = -2 * log_likelihood + 2 * k
            
            # Calculate BIC: -2*log_likelihood + k*ln(n)
            n = len(data)
            bic = -2 * log_likelihood + k * np.log(n)
            
            results.append({
                'Distribution': dist.name,
                'Log-Likelihood': log_likelihood,
                'AIC': aic,
                'BIC': bic,
                'Params': params
            })
        except Exception as e:
            print(f"Error fitting {dist.name}: {str(e)}")
    
    return pd.DataFrame(results).sort_values('AIC')

# Function to plot data with fitted distributions
def plot_with_fitted_distributions(dataset_name, data, top_n=3):
    y = data.iloc[:, -1]
    
    # Fit distributions
    fit_results = fit_distributions(y)
    
    # Plot histogram of data
    plt.figure(figsize=(12, 6))
    sns.histplot(y, kde=False, stat='density', alpha=0.6, label='Data')
    
    # Plot top distributions
    x = np.linspace(min(y), max(y), 1000)
    for i, row in fit_results.head(top_n).iterrows():
        dist_name = row['Distribution']
        dist = getattr(stats, dist_name)
        params = row['Params']
        
        # Plot PDF
        pdf = dist.pdf(x, *params)
        plt.plot(x, pdf, label=f"{dist_name} (AIC: {row['AIC']:.2f})")
    
    plt.title(f"{dataset_name}: Target Distribution with Fitted Distributions")
    plt.xlabel('Target Value')
    plt.ylabel('Density')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
    
    # Print table of results
    print("\nDistribution Fitting Results (sorted by AIC):")
    print(fit_results[['Distribution', 'Log-Likelihood', 'AIC', 'BIC']])
    
    return fit_results

# Analyze each dataset
all_results = {}
for dataset_name, data in datasets_data.items():
    print(f"\n{'='*60}")
    print(f"Analyzing {dataset_name}")
    print(f"{'='*60}")
    all_results[dataset_name] = plot_with_fitted_distributions(dataset_name, data)

## Summary of Best Fitting Distributions

Let's summarize which distribution fits best for each dataset:

In [ ]:
# Create summary table
summary_data = []
for dataset_name, fit_results in all_results.items():
    best_dist = fit_results.iloc[0]
    y = datasets_data[dataset_name].iloc[:, -1]
    
    summary_data.append({
        'Dataset': dataset_name,
        'Best Distribution': best_dist['Distribution'],
        'AIC': best_dist['AIC'],
        'BIC': best_dist['BIC'],
        'Log-Likelihood': best_dist['Log-Likelihood'],
        'Sample Size': len(y),
        'Target Mean': y.mean(),
        'Target Std': y.std(),
        'Target Skewness': y.skew(),
        'Target Kurtosis': y.kurt()
    })

summary_df = pd.DataFrame(summary_data)
summary_df

## Dataset Comparison: Visualization of Properties

Let's create some visualizations to compare key properties across datasets:

In [ ]:
# Plot sample sizes
plt.figure(figsize=(12, 6))
summary_df.sort_values('Sample Size').plot(x='Dataset', y='Sample Size', kind='bar')
plt.title('Sample Size by Dataset')
plt.ylabel('Number of Samples (log scale)')
plt.yscale('log')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Plot skewness and kurtosis
plt.figure(figsize=(12, 6))
summary_df.sort_values('Target Skewness').plot(x='Dataset', y=['Target Skewness', 'Target Kurtosis'], kind='bar')
plt.title('Skewness and Kurtosis by Dataset')
plt.axhline(y=0, color='r', linestyle='-', alpha=0.3)  # Reference line for normal distribution (skewness=0)
plt.axhline(y=3, color='g', linestyle='-', alpha=0.3)  # Reference line for normal distribution (kurtosis=3)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Distribution count
dist_counts = summary_df['Best Distribution'].value_counts()
plt.figure(figsize=(10, 6))
dist_counts.plot(kind='bar')
plt.title('Count of Datasets by Best-Fitting Distribution')
plt.xlabel('Distribution')
plt.ylabel('Count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## Conclusion

This analysis provides insights into the distributional characteristics of the UCI datasets used in LSS-boost experiments. The visualization shows which probability distributions best fit each dataset's target variable, which can help inform modeling choices in probabilistic regression tasks.

Key observations:
1. Most datasets are not perfectly normally distributed
2. Different datasets are best modeled by different probability distributions
3. There is significant variation in sample sizes across datasets
4. Several datasets exhibit significant skewness and non-normal kurtosis

These insights support the use of flexible distribution models like those employed in LSS-boost rather than assuming normality for all datasets.